# Aplyra remote GPU mock test
Run this in a **private** Kaggle notebook with GPU and Internet enabled. It uses a synthetic candidate and submits only to the local mock employer. Do not add your real profile or resume to this notebook.


In [ ]:
import json, os, subprocess, sys, time
from pathlib import Path
from urllib.request import urlopen

root = Path('/kaggle/working/aplyra')
if not root.exists():
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/EvanGruhlkey/Aplyra.git', str(root)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(root / 'requirements.txt'), 'reportlab'], check=True)
subprocess.run([sys.executable, '-m', 'playwright', 'install', '--with-deps', 'chromium'], check=True)
subprocess.run(['apt-get', 'install', '-y', 'zstd'], check=True)
subprocess.run(['bash', '-lc', 'curl -fsSL https://ollama.com/install.sh | sh'], check=True)
subprocess.run(['nvidia-smi'], check=True)


In [ ]:
from reportlab.pdfgen import canvas
profile = {'contact': {'full_name': 'Alex Example', 'email': 'alex@example.com', 'phone': '202-555-0100', 'location': 'Washington, DC', 'linkedin': ''}, 'education': [{'school': 'Example University', 'degree': 'BS Computer Science'}], 'work_experience': [{'company': 'Example Co', 'title': 'Software Intern', 'description': 'Built internal tools'}], 'answers': {'work_authorization': 'Yes', 'why_interested': 'I like building useful software.'}}
profile_path = root / 'profile.json'
resume_path = root / 'resume.pdf'
profile_path.write_text(json.dumps(profile), encoding='utf-8')
pdf = canvas.Canvas(str(resume_path))
pdf.drawString(72, 720, 'Alex Example - synthetic test resume')
pdf.drawString(72, 700, 'Software Intern, Example Co')
pdf.save()


In [ ]:
os.environ['OLLAMA_HOST'] = 'http://127.0.0.1:11434'
os.environ['OLLAMA_MODELS'] = '/kaggle/working/ollama'
os.environ['OLLAMA_MODEL'] = 'qwen3-vl:8b-instruct'
os.environ['OLLAMA_USE_VISION'] = 'true'
os.environ['OLLAMA_NUM_CTX'] = '16384'
os.environ.pop('OLLAMA_NUM_GPU', None)
os.environ['IN_DOCKER'] = 'True'
try:
    urlopen('http://127.0.0.1:11434/api/tags', timeout=2)
except OSError:
    ollama_server = subprocess.Popen(['ollama', 'serve'], env=os.environ.copy(), stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT)
    for _ in range(60):
        try:
            urlopen('http://127.0.0.1:11434/api/tags', timeout=2)
            break
        except OSError:
            time.sleep(1)
    else:
        raise RuntimeError('Ollama did not start')
subprocess.run(['ollama', 'pull', os.environ['OLLAMA_MODEL']], check=True)
subprocess.run(['ollama', 'run', os.environ['OLLAMA_MODEL'], 'Reply READY'], check=True)
subprocess.run(['ollama', 'ps'], check=True)


In [ ]:
sys.path.insert(0, str(root))
from agent import apply, load_profile
mock_server = subprocess.Popen([sys.executable, 'mock_site.py'], cwd=root, stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT)
for _ in range(30):
    try:
        urlopen('http://127.0.0.1:8000/apply', timeout=2)
        break
    except OSError:
        time.sleep(1)
else:
    raise RuntimeError('Mock employer did not start')
run_dir = root / 'runs' / time.strftime('%Y%m%d-%H%M%S')
result = await apply('http://localhost:8000/apply', load_profile(profile_path, resume_path), resume_path, run_dir, headless=True)
print(json.dumps({k: result.get(k) for k in ('status', 'application_id', 'errors', 'screenshot')}, indent=2))
assert result['status'] == 'confirmed'
assert (root / 'mock_submissions' / f"{result['application_id']}.json").exists()
